In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate mlflow evaluate "torchao>=0.16.0"

In [ ]:
import os
import ast
import time
import pickle
import logging
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt   # uses Jupyter's inline backend on Kaggle — plots render in the notebook
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig, PrefixTuningConfig, TaskType,
    get_peft_model, prepare_model_for_kbit_training,
)
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {0: "positive", 1: "neutral", 2: "negative"}
NUM_LABELS = 3

MAX_LENGTH = 128
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 15          # raised from 6 — QLoRA's val F1 was still climbing at epoch 6 in the last
                         # run (never plateaued), so it likely has room left. Early stopping (below)
                         # protects methods like Prefix Tuning that converge/overfit much faster.
LEARNING_RATE = 2e-4     # per spec — higher LR is standard for adapter-only fine-tuning
PATIENCE = 3

TEST_SIZE = 0.15
VAL_SIZE = 0.10

FINBERT_MODEL = "ProsusAI/finbert"
MLFLOW_EXPERIMENT = "IndianFinSentiment"
MLFLOW_MODEL_NAME = "IndianFinSentiment"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. 4-bit QLoRA requires a CUDA GPU — enable one in Kaggle settings.")

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# MLflow 3.x defaults to a SQLite-backed store (metrics/params live INSIDE
# mlflow.db, not as flat files) unless it finds existing file-store data.
# We pin the db to a known path — also required for the Model Registry step later.
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(BASE_DIR, 'mlflow.db')}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATA LOADING — SEntFiN-v1.1 (Indian financial headlines, entity-level sentiment)
# ══════════════════════════════════════════════════════════════

def resolve_sentifin_path() -> str:
    candidates = ["/kaggle/input/datasets/ankurzing/aspect-based-sentiment-analysis-for-financial-news/SEntFiN-v1.1.csv"]
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f == "SEntFiN-v1.1.csv":
                    candidates.insert(0, os.path.join(root, f))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "SEntFiN-v1.1.csv not found. On Kaggle: Add Data -> upload SEntFiN-v1.1.csv as a "
        "dataset and attach it to this notebook. Locally: place it at data/raw/SEntFiN-v1.1.csv."
    )


def load_sentifin() -> pd.DataFrame:
    """
    SEntFiN-v1.1 — 10,753 Indian financial headlines with entity-level sentiment,
    e.g. Decisions = {"SpiceJet": "neutral"}. ~26% of headlines mention >1 entity.
    Since this notebook classifies a whole headline as one label (not entity-aware),
    we keep headlines where every entity shares one sentiment (trivially true for
    single-entity headlines) and drop the ~11.5% where entities disagree — those are
    ambiguous for a single-label classifier.
    """
    log.info("Loading SEntFiN-v1.1...")
    raw = pd.read_csv(resolve_sentifin_path())

    texts, labels, n_entities = [], [], []
    dropped = 0
    for _, row in raw.iterrows():
        try:
            decisions = ast.literal_eval(row["Decisions"])
        except (ValueError, SyntaxError):
            continue
        if not decisions:
            continue
        unique_labels = set(decisions.values())
        if len(unique_labels) != 1:
            dropped += 1
            continue
        texts.append(row["Title"])
        labels.append(unique_labels.pop())
        n_entities.append(len(decisions))

    df = pd.DataFrame({"text": texts, "label": labels, "n_entities": n_entities})
    df["label"] = df["label"].str.lower().str.strip()
    df = df[df["label"].isin(LABEL2ID.keys())]
    df["label_id"] = df["label"].map(LABEL2ID)
    df["source"] = "sentifin"
    log.info(f"  Loaded {len(df)} headlines (dropped {dropped} with conflicting entity sentiment)")
    return df[["text", "label", "label_id", "source", "n_entities"]]


def load_and_combine() -> pd.DataFrame:
    """Load SEntFiN, clean, dedupe. Returns one clean DataFrame."""
    df = load_sentifin()

    df = df.dropna(subset=["text", "label_id"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 10]         # drop near-empty rows
    df = df[df["text"].str.len() < 1000]       # drop unusually long rows
    df = df.drop_duplicates(subset=["text"])

    log.info(f"Combined dataset: {len(df)} samples")
    for label, count in df["label"].value_counts().items():
        log.info(f"  {label:>10}: {count:>6} ({count/len(df)*100:.1f}%)")

    return df.reset_index(drop=True)


def split_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Stratified train/val/test split so class balance holds in every split."""
    train_val_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=VAL_SIZE / (1 - TEST_SIZE),
        stratify=train_val_df["label_id"], random_state=SEED
    )
    log.info(f"Split sizes -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


df_all = load_and_combine()
train_df, val_df, test_df = split_data(df_all)

In [ ]:
hf_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)

def tokenize_df(df: pd.DataFrame):
    enc = hf_tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor(df["label_id"].tolist(), dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)


train_ds = tokenize_df(train_df)
val_ds = tokenize_df(val_df)
test_ds = tokenize_df(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# class weights for the imbalanced 3-class problem
class_counts = train_df["label_id"].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (NUM_LABELS * class_counts), dtype=torch.float32).to(DEVICE)
print(f"Class weights (positive, neutral, negative): {class_weights.tolist()}")

In [ ]:
def compute_metrics(y_true, y_pred) -> dict:
    per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
        "f1_positive": round(per_class[0], 4),
        "f1_neutral": round(per_class[1], 4),
        "f1_negative": round(per_class[2], 4),
    }


def save_confusion_matrix(y_true, y_pred, model_name: str) -> str:
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_pct, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["positive", "neutral", "negative"],
                yticklabels=["positive", "neutral", "negative"], ax=ax, linewidths=0.5)
    ax.set_title(f"Confusion Matrix - {model_name}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    path = os.path.join(RESULTS_DIR, f"cm_{model_name.lower().replace(' ', '_').replace('/', '-')}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


def trainable_param_stats(model) -> dict:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable_params": trainable, "total_params": total, "trainable_pct": round(100 * trainable / total, 4)}


def train_peft_epoch(model, loader, optimizer, criterion, scheduler=None) -> float:
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_peft(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        total_loss += loss.item() * input_ids.size(0)

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, compute_metrics(all_labels, all_preds), all_preds, all_labels


def fit_peft(model, model_name: str, epochs=NUM_EPOCHS, lr=LEARNING_RATE, patience=PATIENCE):
    """Same early-stopping-on-val-F1 pattern used for Models 3-6, adapted to HF's forward() signature.

    Uses a linear warmup + linear decay learning rate schedule (10% warmup) — standard for
    transformer fine-tuning, and it directly addresses what we saw in the last run: Prefix Tuning's
    val F1 peaked at epoch 1 and got steadily worse afterward under a flat learning rate, which is
    a classic sign the LR stayed too high for too long. Decaying it over training gives every
    variant a chance to settle into a better minimum instead of overshooting it.

    NOTE: the best-epoch checkpoint only snapshots/restores the TRAINABLE parameters (the
    LoRA/prefix adapter weights + classifier head), not the full model.state_dict(). For QLoRA,
    the frozen base model is 4-bit quantized via bitsandbytes, which stores extra per-weight
    buffers (absmax, quant_map, nested_absmax, ...) alongside each quantized tensor — those don't
    round-trip cleanly through state_dict()/load_state_dict() (cloning + reloading them raises
    "Unexpected key(s) in state_dict"). Since the base model is frozen and never changes during
    training, only the trainable adapter parameters ever need to be checkpointed anyway.
    """
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_f1, best_state, epochs_no_improve = -1.0, None, 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = train_peft_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(model, val_loader, criterion)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        for k, v in val_metrics.items():
            mlflow.log_metric(f"val_{k}", v, step=epoch)

        print(f"[{model_name}] Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_metrics['f1_weighted']:.4f}")

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    train_time = time.time() - start
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))
    return model, train_time, best_val_f1

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Without this, the classifier head (freshly initialized for our 3-class problem,
    # not part of FinBERT's pretrained weights) also gets quantized into a non-floating-point
    # Linear4bit layer. PEFT then can't set requires_grad=True on it, and get_peft_model()
    # crashes with "only Tensors of floating point dtype can require gradients".
    llm_int8_skip_modules=["classifier"],
)

base_model_qlora = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_MODEL, num_labels=NUM_LABELS, quantization_config=bnb_config, device_map={"": 0}
)
base_model_qlora = prepare_model_for_kbit_training(base_model_qlora)

qlora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
qlora_model = get_peft_model(base_model_qlora, qlora_config)
qlora_model.print_trainable_parameters()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="FinBERT-QLoRA"):
    mlflow.log_params({
        "peft_method": "QLoRA", "quant": "4bit-nf4", "r": 8, "lora_alpha": 16,
        "target_modules": "query,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    qlora_model, qlora_time, qlora_val_f1 = fit_peft(qlora_model, "QLoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        qlora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(qlora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(qlora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-QLoRA")
    mlflow.log_artifact(cm_path)

    qlora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_qlora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": qlora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    qlora_run_id = mlflow.active_run().info.run_id

qlora_results = {"name": "QLoRA (r=8, q+v)", "test_metrics": test_metrics, "stats": stats,
                  "train_time": qlora_time, "run_id": qlora_run_id}
print(qlora_results)

In [ ]:
base_model_lora = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL, num_labels=NUM_LABELS).to(DEVICE)

lora_config = LoraConfig(
    r=16, lora_alpha=16, target_modules=["query", "key", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(base_model_lora, lora_config)
lora_model.print_trainable_parameters()

with mlflow.start_run(run_name="FinBERT-LoRA"):
    mlflow.log_params({
        "peft_method": "LoRA", "quant": "none", "r": 16, "lora_alpha": 16,
        "target_modules": "query,key,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    lora_model, lora_time, lora_val_f1 = fit_peft(lora_model, "LoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        lora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(lora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(lora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA")
    mlflow.log_artifact(cm_path)

    lora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": lora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    lora_run_id = mlflow.active_run().info.run_id

lora_results = {"name": "LoRA (r=16, q+k+v)", "test_metrics": test_metrics, "stats": stats,
                 "train_time": lora_time, "run_id": lora_run_id}
print(lora_results)

In [ ]:
# ══════════════════════════════════════════════════════════════
# SEARCH SPACE + HELPERS
# ══════════════════════════════════════════════════════════════
import random
from transformers import get_cosine_schedule_with_warmup

SEARCH_SEED = 123

SEARCH_SPACE = {
    "learning_rate": [1e-4, 2e-4, 3e-4, 5e-4],
    "lora_r": [8, 16, 32],
    "lora_alpha_mult": [1, 2],                 # lora_alpha = lora_r * this
    "target_modules": [
        ("query", "value"),
        ("query", "key", "value"),
        ("query", "key", "value", "dense"),
    ],
    "lora_dropout": [0.05, 0.1, 0.2],
    "max_length": [32, 48, 64],                 # was a fixed 128 — see markdown above
    "batch_size": [16, 32, 64],
    "weight_decay": [0.0, 0.01, 0.1],
    "optimizer_name": ["adamw", "adam", "rmsprop"],
    "scheduler_type": ["linear", "cosine"],
    "warmup_ratio": [0.06, 0.1],
    "label_smoothing": [0.0, 0.05, 0.1],
    "unfreeze_last_n_layers": [0, 1, 2],        # 0 = pure LoRA, like the baseline run above
}


def sample_config(rng: random.Random) -> dict:
    return {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}


def build_loaders(max_length: int, batch_size: int):
    """Re-tokenize at a given max_length and wrap in DataLoaders at a given batch_size.
    Cheap enough to redo per trial — tokenizing ~9.5k short headlines takes well under a second."""
    def _tok(df):
        enc = hf_tokenizer(
            df["text"].tolist(), padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        labels = torch.tensor(df["label_id"].tolist(), dtype=torch.long)
        return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

    tr = DataLoader(_tok(train_df), batch_size=batch_size, shuffle=True)
    va = DataLoader(_tok(val_df), batch_size=batch_size, shuffle=False)
    te = DataLoader(_tok(test_df), batch_size=batch_size, shuffle=False)
    return tr, va, te


def build_optimizer(name: str, params, lr: float, weight_decay: float):
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(kind: str, optimizer, warmup_steps: int, total_steps: int):
    if kind == "linear":
        return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    if kind == "cosine":
        return get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    raise ValueError(f"Unknown scheduler: {kind}")


def apply_layer_unfreezing(model, n_layers: int):
    """Fully unfreeze the last n transformer blocks (on top of whatever LoRA already adapts).
    Matches on the 'encoder.layer.{i}.' substring, which survives PEFT's module-path wrapping
    regardless of exact prefix (e.g. base_model.model.bert.encoder.layer.11...)."""
    if n_layers <= 0:
        return
    total_layers = model.config.num_hidden_layers if hasattr(model, "config") else 12
    unfreeze_from = total_layers - n_layers
    target_substrings = [f"encoder.layer.{i}." for i in range(unfreeze_from, total_layers)]
    for name, param in model.named_parameters():
        if any(sub in name for sub in target_substrings):
            param.requires_grad = True

In [ ]:
# ══════════════════════════════════════════════════════════════
# TRIAL RUNNER — trains one config, returns its best val F1 + the trained model
# ══════════════════════════════════════════════════════════════

def run_lora_trial(config: dict, epochs: int, patience: int, log_to_mlflow: bool = True):
    tr_loader, va_loader, te_loader = build_loaders(config["max_length"], config["batch_size"])

    base_model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL, num_labels=NUM_LABELS
    ).to(DEVICE)

    lora_alpha = config["lora_r"] * config["lora_alpha_mult"]
    lora_cfg = LoraConfig(
        r=config["lora_r"], lora_alpha=lora_alpha,
        target_modules=list(config["target_modules"]),
        lora_dropout=config["lora_dropout"], bias="none",
        task_type=TaskType.SEQ_CLS, modules_to_save=["classifier"],
    )
    model = get_peft_model(base_model, lora_cfg)
    apply_layer_unfreezing(model, config["unfreeze_last_n_layers"])

    optimizer = build_optimizer(
        config["optimizer_name"], [p for p in model.parameters() if p.requires_grad],
        config["learning_rate"], config["weight_decay"],
    )
    total_steps = len(tr_loader) * epochs
    warmup_steps = int(config["warmup_ratio"] * total_steps)
    scheduler = build_scheduler(config["scheduler_type"], optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config["label_smoothing"])

    best_val_f1, best_state, epochs_no_improve = -1.0, None, 0
    start = time.time()
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_peft_epoch(model, tr_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(model, va_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **val_metrics})

        if log_to_mlflow:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            for k, v in val_metrics.items():
                mlflow.log_metric(f"val_{k}", v, step=epoch)

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    train_time = time.time() - start
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))

    return {
        "model": model, "best_val_f1": best_val_f1, "history": history,
        "train_time": train_time, "test_loader": te_loader, "criterion": criterion,
        "config": config,
    }

In [ ]:
# ══════════════════════════════════════════════════════════════
# STAGE 1 — random search (short budget per trial, just to RANK configs)
# Retains the actual trained model of whichever trial holds the best val F1 so far,
# instead of discarding every trial's weights — see markdown below for why this matters.
# ══════════════════════════════════════════════════════════════
N_TRIALS = 12          # raise for a more thorough search — cost scales linearly
SEARCH_EPOCHS = 5      # short budget: enough to see which configs are promising, not to converge
SEARCH_PATIENCE = 2

rng = random.Random(SEARCH_SEED)
trial_configs = [sample_config(rng) for _ in range(N_TRIALS)]

leaderboard = []
best_overall = {"val_f1": -1.0, "model": None, "config": None,
                 "test_loader": None, "criterion": None}
mlflow.set_experiment(MLFLOW_EXPERIMENT)

for i, cfg in enumerate(trial_configs, start=1):
    print(f"\n=== Trial {i}/{N_TRIALS} ===")
    print(cfg)
    with mlflow.start_run(run_name=f"LoRA-HPT-trial-{i}"):
        loggable_cfg = {**cfg, "target_modules": ",".join(cfg["target_modules"])}
        mlflow.log_params(loggable_cfg)

        result = run_lora_trial(cfg, epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE)
        mlflow.log_metric("best_val_f1", result["best_val_f1"])

        leaderboard.append({
            "trial": i, **loggable_cfg,
            "best_val_f1": result["best_val_f1"],
            "train_time": round(result["train_time"], 1),
        })

    if result["best_val_f1"] > best_overall["val_f1"]:
        if best_overall["model"] is not None:
            del best_overall["model"]
            torch.cuda.empty_cache()
        best_overall = {
            "val_f1": result["best_val_f1"], "model": result["model"], "config": cfg,
            "test_loader": result["test_loader"], "criterion": result["criterion"],
        }
    else:
        del result["model"]
        torch.cuda.empty_cache()

leaderboard_df = pd.DataFrame(leaderboard).sort_values("best_val_f1", ascending=False).reset_index(drop=True)
leaderboard_df.to_csv(os.path.join(RESULTS_DIR, "lora_hpt_leaderboard.csv"), index=False)
print(f"\nSearch done — total time: {leaderboard_df['train_time'].sum() / 60:.1f} min")
print(f"Best trial retained in memory: val_f1={best_overall['val_f1']:.4f}")
leaderboard_df

## 10. Continue training the actual best trial (not a fresh reinit)

**What happened in the run above:** the winning config reached **val F1 = 0.899** after just 5
search-stage epochs. But the "Stage 2" cell further down takes that config and rebuilds the model
**from scratch** (`AutoModelForSequenceClassification.from_pretrained(...)` + a freshly
re-initialized LoRA adapter and classifier head), then retrains for up to 15 epochs. That run
finished at **test F1 = 0.8590 / accuracy = 0.8591 — *worse* than the untuned baseline
(0.8652 / 0.8654)**.

That's a real, useful result, not a bug: it shows two things —
1. **Reinitializing throws away the good state the search already found.** The 5-epoch run
   reached 0.899 val F1 with one specific random draw of LoRA's A/B matrices and the classifier
   head; a second "from scratch" run of the *same config* draws different random weights and
   isn't guaranteed to retrace the same path, especially over 3x more epochs.
2. **Picking a "winner" from 12 short runs against a 946-row validation set is somewhat noisy** —
   the true best config and the best-looking config on this particular sample aren't guaranteed
   to be identical, but that's a separate issue from (1).

The fix for (1) is straightforward: don't throw the trained weights away. The search loop above now
keeps the actual model object of whichever trial holds the best val F1 (`best_overall["model"]`) —
this cell continues optimizing *that exact model* with a fresh optimizer/scheduler (starting at a
lower LR, since we're extending training near a good point rather than starting cold) instead of
rebuilding from a new random init.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Continue training best_overall["model"] — no reinitialization
# ══════════════════════════════════════════════════════════════
continued_model = best_overall["model"]
continued_config = best_overall["config"]
te_loader = best_overall["test_loader"]
criterion = best_overall["criterion"]

CONTINUE_EPOCHS = 15
CONTINUE_PATIENCE = 5
# Halve the LR for the continuation phase: we're extending training from a point that's
# already good, not starting cold, so the aggressive search-stage LR is more likely to
# knock the model out of its current optimum than to find a meaningfully better one.
CONTINUE_LR = continued_config["learning_rate"] * 0.5

optimizer = build_optimizer(
    continued_config["optimizer_name"],
    [p for p in continued_model.parameters() if p.requires_grad],
    CONTINUE_LR, continued_config["weight_decay"],
)
# Rebuild loaders at THIS config's max_length/batch_size — the model was trained on these
# during the search stage, not the notebook-level train_loader/val_loader (MAX_LENGTH=128).
tr_loader, va_loader, _ = build_loaders(continued_config["max_length"], continued_config["batch_size"])
total_steps = len(tr_loader) * CONTINUE_EPOCHS
warmup_steps = int(continued_config["warmup_ratio"] * total_steps)
scheduler = build_scheduler(continued_config["scheduler_type"], optimizer, warmup_steps, total_steps)

best_val_f1 = best_overall["val_f1"]
best_state = {name: p.detach().cpu().clone() for name, p in continued_model.named_parameters() if p.requires_grad}
epochs_no_improve = 0
start = time.time()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="FinBERT-LoRA-Continued"):
    loggable_cfg = {**continued_config, "target_modules": ",".join(continued_config["target_modules"]),
                     "continue_lr": CONTINUE_LR}
    mlflow.log_params(loggable_cfg)
    mlflow.log_metric("val_f1_at_start", best_val_f1)

    for epoch in range(1, CONTINUE_EPOCHS + 1):
        train_loss = train_peft_epoch(continued_model, tr_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(continued_model, va_loader, criterion)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        for k, v in val_metrics.items():
            mlflow.log_metric(f"val_{k}", v, step=epoch)

        print(f"[Continued] Epoch {epoch}/{CONTINUE_EPOCHS} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_metrics['f1_weighted']:.4f}")

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in continued_model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONTINUE_PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    continue_time = time.time() - start
    with torch.no_grad():
        for name, param in continued_model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(continued_model, te_loader, criterion)
    stats = trainable_param_stats(continued_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(continue_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA-Continued")
    mlflow.log_artifact(cm_path)

    continued_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_continued_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": continued_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    continued_run_id = mlflow.active_run().info.run_id

print(f"\n{'='*55}\n  FinBERT-LoRA-Continued — TEST RESULTS\n{'='*55}")
print(f"  Accuracy    : {test_metrics['accuracy']:.4f}")
print(f"  F1 Weighted : {test_metrics['f1_weighted']:.4f}")
print(f"  F1 Positive : {test_metrics['f1_positive']:.4f}")
print(f"  F1 Neutral  : {test_metrics['f1_neutral']:.4f}")
print(f"  F1 Negative : {test_metrics['f1_negative']:.4f}")
print(f"{'='*55}")
print(classification_report(test_labels, test_preds, target_names=["positive", "neutral", "negative"]))

continued_results = {"name": "LoRA-Continued", "test_metrics": test_metrics, "stats": stats,
                      "train_time": continue_time, "run_id": continued_run_id, "config": continued_config}
print(continued_results)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Pick the winning config, retrain fully, evaluate test set ONCE
# ══════════════════════════════════════════════════════════════
best_row = leaderboard_df.iloc[0]
best_config = {k: best_row[k] for k in SEARCH_SPACE.keys()}
best_config["target_modules"] = tuple(best_row["target_modules"].split(","))
# leaderboard_df stores numeric fields as numpy scalars — plain python types are safer downstream
for k in ["lora_r", "lora_alpha_mult", "max_length", "batch_size", "unfreeze_last_n_layers"]:
    best_config[k] = int(best_config[k])
for k in ["learning_rate", "lora_dropout", "weight_decay", "warmup_ratio", "label_smoothing"]:
    best_config[k] = float(best_config[k])

print("Best config from search (val F1 =", best_row["best_val_f1"], "):")
print(best_config)

FINAL_EPOCHS = 15
FINAL_PATIENCE = 5   # more patient than the original run's 3 — LoRA's val F1 was still (slowly)
                     # improving through epoch 13 last time, so 3 risked stopping right as it
                     # was still finding gains. Extra epochs are cheap once we've narrowed to
                     # a single config instead of paying this cost across every search trial.

with mlflow.start_run(run_name="FinBERT-LoRA-Tuned"):
    loggable_best = {**best_config, "target_modules": ",".join(best_config["target_modules"])}
    mlflow.log_params(loggable_best)
    mlflow.log_param("epochs_budget", FINAL_EPOCHS)
    mlflow.log_param("patience", FINAL_PATIENCE)

    final_result = run_lora_trial(best_config, epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE)
    tuned_model = final_result["model"]

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        tuned_model, final_result["test_loader"], final_result["criterion"]
    )
    stats = trainable_param_stats(tuned_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(final_result["train_time"], 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA-Tuned")
    mlflow.log_artifact(cm_path)

    tuned_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_tuned_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": tuned_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    tuned_run_id = mlflow.active_run().info.run_id

print(f"\n{'='*55}\n  FinBERT-LoRA-Tuned — TEST RESULTS\n{'='*55}")
print(f"  Accuracy    : {test_metrics['accuracy']:.4f}")
print(f"  F1 Weighted : {test_metrics['f1_weighted']:.4f}")
print(f"  F1 Positive : {test_metrics['f1_positive']:.4f}")
print(f"  F1 Neutral  : {test_metrics['f1_neutral']:.4f}")
print(f"  F1 Negative : {test_metrics['f1_negative']:.4f}")
print(f"{'='*55}")
print(classification_report(test_labels, test_preds, target_names=["positive", "neutral", "negative"]))

tuned_results = {"name": "LoRA-Tuned", "test_metrics": test_metrics, "stats": stats,
                  "train_time": final_result["train_time"], "run_id": tuned_run_id,
                  "config": best_config}
print(tuned_results)

In [ ]:
print(f"{'Method':<24} {'F1 (weighted)':>14} {'Accuracy':>10}")
print("-" * 52)
print(f"{'LoRA (r=16, baseline)':<24} {lora_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{lora_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Tuned (reinit)':<24} {tuned_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{tuned_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Continued':<24} {continued_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{continued_results['test_metrics']['accuracy']:>10.4f}")

candidates = [
    ("baseline", lora_results),
    ("tuned (reinit)", tuned_results),
    ("continued", continued_results),
]
winner_label, winner = max(candidates, key=lambda kv: kv[1]["test_metrics"]["f1_weighted"])
print(f"\nOverall winner: {winner_label} (F1={winner['test_metrics']['f1_weighted']:.4f})")

if winner_label != "baseline":
    print("A tuned/continued run beat the original baseline — registering it as Production.")
    client = MlflowClient()
    model_uri = f"runs:/{winner['run_id']}/model"
    try:
        registered = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
        client.set_registered_model_alias(name=MLFLOW_MODEL_NAME, alias="Production", version=registered.version)
        print(f"Registered '{winner_label}' as '{MLFLOW_MODEL_NAME}' v{registered.version} -> alias=Production")
    except Exception as e:
        print(f"Registry step skipped/failed ({e}). Adapter files are still saved locally.")
else:
    print("Neither tuned run beat the untuned baseline — keeping the baseline as Production.")
    print("Consider raising N_TRIALS for a wider search, or manually trying the runner-up configs "
          "in lora_hpt_leaderboard.csv.")

print(f"\nDownload these from the Kaggle notebook's Output tab before your session ends:")
print(f"  - {MODELS_DIR}/finbert_lora_tuned_adapter/")
print(f"  - {MODELS_DIR}/finbert_lora_continued_adapter/")
print(f"  - {RESULTS_DIR}/lora_hpt_leaderboard.csv")
print(f"  - {os.path.join(BASE_DIR, 'mlflow.db')}")

## 12. Download everything

Zips every adapter, result, and MLflow log into one archive so you don't have to click through
the Output tab file-by-file.

In [ ]:
# ══════════════════════════════════════════════════════════════
# DOWNLOAD EVERYTHING — one zip with every adapter, plot, CSV, and MLflow log
# ══════════════════════════════════════════════════════════════
import shutil
from IPython.display import FileLink, display

# The tokenizer is shared by every variant (QLoRA/LoRA/Tuned/Continued all use FinBERT's own
# WordPiece vocab) and is only saved inline inside each mlflow.transformers.log_model bundle so
# far — save it once here too so it's easy to grab standalone.
hf_tokenizer.save_pretrained(os.path.join(MODELS_DIR, "finbert_tokenizer"))

ARCHIVE_NAME = "finbert_peft_all_artifacts"
ARCHIVE_STAGING = os.path.join(BASE_DIR, ARCHIVE_NAME)
os.makedirs(ARCHIVE_STAGING, exist_ok=True)

# Copy (not move) each artifact directory/file into a staging folder, skipping anything missing —
# not every variant necessarily ran in every session (e.g. QLoRA needs CUDA).
items_to_include = [
    MODELS_DIR,                                    # every adapter + shared tokenizer
    RESULTS_DIR,                                   # confusion matrices, leaderboard CSV, plots
    os.path.join(BASE_DIR, "mlflow.db"),           # all logged params/metrics
    os.path.join(BASE_DIR, "mlruns"),              # MLflow artifact store (models, cm images)
]

for src in items_to_include:
    if not os.path.exists(src):
        print(f"Skipping (not found): {src}")
        continue
    dest = os.path.join(ARCHIVE_STAGING, os.path.basename(src))
    if os.path.isdir(src):
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)
    print(f"Added: {src}")

zip_path = shutil.make_archive(
    base_name=os.path.join(BASE_DIR, ARCHIVE_NAME),   # -> {ARCHIVE_NAME}.zip
    format="zip",
    root_dir=ARCHIVE_STAGING,
)
shutil.rmtree(ARCHIVE_STAGING)   # remove the staging copy, keep only the zip

size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"\nCreated {zip_path} ({size_mb:.1f} MB)")
print("Contents: saved_models/ (all adapters + tokenizer), results/ (plots, leaderboard CSV), "
      "mlflow.db, mlruns/")

# On Kaggle, this renders as a clickable one-shot download link in the notebook output.
# If it doesn't render (e.g. running locally), the file still exists at zip_path —
# grab it from the Kaggle Output tab, or `zip_path` directly if running locally.
try:
    display(FileLink(zip_path))
except Exception as e:
    print(f"FileLink didn't render ({e}). Download {zip_path} from the Output tab instead.")

## 13. One more try — narrowed search, avoiding the config that kept losing

Two independent runs of the search above (sections 9-11) picked the same kind of "winner":
high learning rate (`5e-4`) combined with `unfreeze_last_n_layers=2` — fully unfreezing 2
FinBERT transformer blocks on top of LoRA, which pushes trainable params from the baseline's
0.8% up to 13.3%. Both times that config looked great on the 5-epoch/946-row validation proxy
(val F1 ≈ 0.899) and both times it failed to beat the plain untuned baseline once evaluated
properly (0.8633 and 0.8617 vs. the baseline's 0.8652). That's not noise anymore — it's a
reproducible failure mode: high capacity + high LR overfits the small validation set's
particular sample fast, without the gains generalizing to the test set.

This retry removes exactly those two culprits from the search space (`5e-4` and
`unfreeze_last_n_layers=2`) and gives each trial more epochs to prove itself before being crowned
a winner (8 instead of 5) — a short budget is exactly what let an overfitting-prone config "win"
by spiking fast last time. Everything else (rank, alpha, target modules, dropout, max_length,
batch size, weight decay, optimizer, scheduler, warmup, label smoothing) stays in play.

In [ ]:
# ══════════════════════════════════════════════════════════════
# RETRY SEARCH SPACE — drops the two settings implicated in both losing "winners"
# ══════════════════════════════════════════════════════════════
SEARCH_SPACE_V2 = {
    "learning_rate": [1e-4, 2e-4, 3e-4],         # dropped 5e-4
    "lora_r": [8, 16, 32],
    "lora_alpha_mult": [1, 2],
    "target_modules": [
        ("query", "value"),
        ("query", "key", "value"),
        ("query", "key", "value", "dense"),
    ],
    "lora_dropout": [0.05, 0.1, 0.2],
    "max_length": [32, 48, 64],
    "batch_size": [16, 32, 64],
    "weight_decay": [0.0, 0.01, 0.1],
    "optimizer_name": ["adamw", "adam", "rmsprop"],
    "scheduler_type": ["linear", "cosine"],
    "warmup_ratio": [0.06, 0.1],
    "label_smoothing": [0.0, 0.05, 0.1],
    "unfreeze_last_n_layers": [0, 1],            # dropped 2
}

# sample_config()/run_lora_trial() both read the *global* SEARCH_SPACE by name, so pointing it
# at the narrowed space is enough to reuse every helper from sections 9-11 unchanged.
SEARCH_SPACE = SEARCH_SPACE_V2

N_TRIALS_V2 = 10
SEARCH_EPOCHS_V2 = 8       # up from 5 — harder for a fast-but-shallow config to fake a win
SEARCH_PATIENCE_V2 = 3

rng_v2 = random.Random(SEARCH_SEED + 1)   # new seed — SEARCH_SEED's draws are already spent
trial_configs_v2 = [sample_config(rng_v2) for _ in range(N_TRIALS_V2)]

leaderboard_v2 = []
best_overall_v2 = {"val_f1": -1.0, "model": None, "config": None,
                    "test_loader": None, "criterion": None}
mlflow.set_experiment(MLFLOW_EXPERIMENT)

for i, cfg in enumerate(trial_configs_v2, start=1):
    print(f"\n=== Retry Trial {i}/{N_TRIALS_V2} ===")
    print(cfg)
    with mlflow.start_run(run_name=f"LoRA-HPT-v2-trial-{i}"):
        loggable_cfg = {**cfg, "target_modules": ",".join(cfg["target_modules"])}
        mlflow.log_params(loggable_cfg)

        result = run_lora_trial(cfg, epochs=SEARCH_EPOCHS_V2, patience=SEARCH_PATIENCE_V2)
        mlflow.log_metric("best_val_f1", result["best_val_f1"])

        leaderboard_v2.append({
            "trial": i, **loggable_cfg,
            "best_val_f1": result["best_val_f1"],
            "train_time": round(result["train_time"], 1),
        })

    if result["best_val_f1"] > best_overall_v2["val_f1"]:
        if best_overall_v2["model"] is not None:
            del best_overall_v2["model"]
            torch.cuda.empty_cache()
        best_overall_v2 = {
            "val_f1": result["best_val_f1"], "model": result["model"], "config": cfg,
            "test_loader": result["test_loader"], "criterion": result["criterion"],
        }
    else:
        del result["model"]
        torch.cuda.empty_cache()

leaderboard_v2_df = pd.DataFrame(leaderboard_v2).sort_values("best_val_f1", ascending=False).reset_index(drop=True)
leaderboard_v2_df.to_csv(os.path.join(RESULTS_DIR, "lora_hpt_leaderboard_v2.csv"), index=False)
print(f"\nRetry search done — total time: {leaderboard_v2_df['train_time'].sum() / 60:.1f} min")
print(f"Best retry trial retained: val_f1={best_overall_v2['val_f1']:.4f}")
leaderboard_v2_df

## 14. Continue training the retry's best trial (same no-reinit approach as section 10)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Continue training best_overall_v2["model"] — no reinitialization (same pattern as section 10)
# ══════════════════════════════════════════════════════════════
continued_model_v2 = best_overall_v2["model"]
continued_config_v2 = best_overall_v2["config"]
te_loader_v2 = best_overall_v2["test_loader"]
criterion_v2 = best_overall_v2["criterion"]

CONTINUE_EPOCHS_V2 = 15
CONTINUE_PATIENCE_V2 = 5
CONTINUE_LR_V2 = continued_config_v2["learning_rate"] * 0.5

optimizer = build_optimizer(
    continued_config_v2["optimizer_name"],
    [p for p in continued_model_v2.parameters() if p.requires_grad],
    CONTINUE_LR_V2, continued_config_v2["weight_decay"],
)
tr_loader, va_loader, _ = build_loaders(continued_config_v2["max_length"], continued_config_v2["batch_size"])
total_steps = len(tr_loader) * CONTINUE_EPOCHS_V2
warmup_steps = int(continued_config_v2["warmup_ratio"] * total_steps)
scheduler = build_scheduler(continued_config_v2["scheduler_type"], optimizer, warmup_steps, total_steps)

best_val_f1 = best_overall_v2["val_f1"]
best_state = {name: p.detach().cpu().clone() for name, p in continued_model_v2.named_parameters() if p.requires_grad}
epochs_no_improve = 0
start = time.time()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="FinBERT-LoRA-Continued-v2"):
    loggable_cfg = {**continued_config_v2, "target_modules": ",".join(continued_config_v2["target_modules"]),
                     "continue_lr": CONTINUE_LR_V2}
    mlflow.log_params(loggable_cfg)
    mlflow.log_metric("val_f1_at_start", best_val_f1)

    for epoch in range(1, CONTINUE_EPOCHS_V2 + 1):
        train_loss = train_peft_epoch(continued_model_v2, tr_loader, optimizer, criterion_v2, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(continued_model_v2, va_loader, criterion_v2)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        for k, v in val_metrics.items():
            mlflow.log_metric(f"val_{k}", v, step=epoch)

        print(f"[Continued-v2] Epoch {epoch}/{CONTINUE_EPOCHS_V2} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_metrics['f1_weighted']:.4f}")

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in continued_model_v2.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONTINUE_PATIENCE_V2:
                print(f"Early stopping at epoch {epoch}.")
                break

    continue_time_v2 = time.time() - start
    with torch.no_grad():
        for name, param in continued_model_v2.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(continued_model_v2, te_loader_v2, criterion_v2)
    stats = trainable_param_stats(continued_model_v2)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(continue_time_v2, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA-Continued-v2")
    mlflow.log_artifact(cm_path)

    continued_model_v2.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_continued_v2_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": continued_model_v2, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    continued_v2_run_id = mlflow.active_run().info.run_id

print(f"\n{'='*55}\n  FinBERT-LoRA-Continued-v2 — TEST RESULTS\n{'='*55}")
print(f"  Accuracy    : {test_metrics['accuracy']:.4f}")
print(f"  F1 Weighted : {test_metrics['f1_weighted']:.4f}")
print(f"  F1 Positive : {test_metrics['f1_positive']:.4f}")
print(f"  F1 Neutral  : {test_metrics['f1_neutral']:.4f}")
print(f"  F1 Negative : {test_metrics['f1_negative']:.4f}")
print(f"{'='*55}")
print(classification_report(test_labels, test_preds, target_names=["positive", "neutral", "negative"]))

continued_results_v2 = {"name": "LoRA-Continued-v2", "test_metrics": test_metrics, "stats": stats,
                         "train_time": continue_time_v2, "run_id": continued_v2_run_id,
                         "config": continued_config_v2}
print(continued_results_v2)

In [ ]:
# ══════════════════════════════════════════════════════════════
# FINAL comparison across every candidate tried, register the true winner
# ══════════════════════════════════════════════════════════════
print(f"{'Method':<28} {'F1 (weighted)':>14} {'Accuracy':>10}")
print("-" * 56)
print(f"{'LoRA (r=16, baseline)':<28} {lora_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{lora_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Tuned (reinit, v1)':<28} {tuned_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{tuned_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Continued (v1)':<28} {continued_results['test_metrics']['f1_weighted']:>14.4f} "
      f"{continued_results['test_metrics']['accuracy']:>10.4f}")
print(f"{'LoRA-Continued (v2 retry)':<28} {continued_results_v2['test_metrics']['f1_weighted']:>14.4f} "
      f"{continued_results_v2['test_metrics']['accuracy']:>10.4f}")

candidates = [
    ("baseline", lora_results),
    ("tuned v1 (reinit)", tuned_results),
    ("continued v1", continued_results),
    ("continued v2 (narrowed retry)", continued_results_v2),
]
winner_label, winner = max(candidates, key=lambda kv: kv[1]["test_metrics"]["f1_weighted"])
print(f"\nOverall winner: {winner_label} (F1={winner['test_metrics']['f1_weighted']:.4f})")

if winner_label != "baseline":
    print("A tuned/continued run beat the original baseline — registering it as Production.")
    client = MlflowClient()
    model_uri = f"runs:/{winner['run_id']}/model"
    try:
        registered = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
        client.set_registered_model_alias(name=MLFLOW_MODEL_NAME, alias="Production", version=registered.version)
        print(f"Registered '{winner_label}' as '{MLFLOW_MODEL_NAME}' v{registered.version} -> alias=Production")
    except Exception as e:
        print(f"Registry step skipped/failed ({e}). Adapter files are still saved locally.")
else:
    print("Nothing beat the untuned baseline across either search — keeping the baseline as Production.")
    print("At this point the untuned LoRA config (r=16, q+k+v, max_length=128, lr=2e-4) is the")
    print("strongest reproducible result for FinBERT+PEFT on this dataset.")

print(f"\nAll adapters saved under {MODELS_DIR}/:")
print(f"  - finbert_lora_adapter/            (baseline)")
print(f"  - finbert_lora_tuned_adapter/       (v1 reinit)")
print(f"  - finbert_lora_continued_adapter/   (v1 continued)")
print(f"  - finbert_lora_continued_v2_adapter/ (v2 narrowed retry, continued)")

## 15. Re-download everything (includes the v2 retry artifacts)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DOWNLOAD EVERYTHING (v2) — same as section 12, rerun so the zip includes the retry's artifacts
# ══════════════════════════════════════════════════════════════
ARCHIVE_NAME_V2 = "finbert_peft_all_artifacts_v2"
ARCHIVE_STAGING_V2 = os.path.join(BASE_DIR, ARCHIVE_NAME_V2)
os.makedirs(ARCHIVE_STAGING_V2, exist_ok=True)

items_to_include = [
    MODELS_DIR,
    RESULTS_DIR,
    os.path.join(BASE_DIR, "mlflow.db"),
    os.path.join(BASE_DIR, "mlruns"),
]

for src in items_to_include:
    if not os.path.exists(src):
        print(f"Skipping (not found): {src}")
        continue
    dest = os.path.join(ARCHIVE_STAGING_V2, os.path.basename(src))
    if os.path.isdir(src):
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)
    print(f"Added: {src}")

zip_path_v2 = shutil.make_archive(
    base_name=os.path.join(BASE_DIR, ARCHIVE_NAME_V2),
    format="zip",
    root_dir=ARCHIVE_STAGING_V2,
)
shutil.rmtree(ARCHIVE_STAGING_V2)

size_mb = os.path.getsize(zip_path_v2) / (1024 ** 2)
print(f"\nCreated {zip_path_v2} ({size_mb:.1f} MB)")
print("Contents: every adapter (baseline, v1 tuned, v1 continued, v2 continued) + tokenizer, "
      "results/ (both leaderboards + plots), mlflow.db, mlruns/")

try:
    display(FileLink(zip_path_v2))
except Exception as e:
    print(f"FileLink didn't render ({e}). Download {zip_path_v2} from the Output tab instead.")